# Sex & Individual ID Training Workflow

Dieses Notebook zeigt, wie sich die FIT-Pipeline sowohl für die Geschlechtsklassifikation als auch für die Individual-ID-Modelle ausführen lässt. Zusätzlich werden die neuen Unsicherheits-Auswertungen eingebunden, sodass Kalibrierung und Vertrauensbereiche direkt überprüft werden können.

## 1. Umgebung vorbereiten

Wir definieren einen eigenen Ergebnisordner und laden die globale Konfiguration. Durch das Setzen der Umgebungsvariablen lassen sich alle Dateipfade kontrolliert in einen Notebook-spezifischen Unterordner lenken.

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import os
from importlib import reload

EXPERIMENT_NAME = f'sex_id_notebook_{datetime.now():%Y%m%d_%H%M%S}'
BASE_RESULTS_DIR = Path('results/notebook_runs')
EXPERIMENT_ROOT = (BASE_RESULTS_DIR / EXPERIMENT_NAME).resolve()
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

os.environ['FIT_EXPERIMENT_ROOT'] = str(EXPERIMENT_ROOT)
os.environ['FIT_EXPERIMENT_NAME'] = EXPERIMENT_NAME

import FIT_python.config as config
reload(config)

print('Experiment root:', config.EXPERIMENT_ROOT)
print('Raw data directory:', config.RAW_DIR)
print('Unified results directory:', config.RESULTS_DIR)

Für interaktive Analysen aktivieren wir automatische Modul-Neuladungen und konfigurieren das Plot-Layout.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')

## 2. Zentrale Konfigurationsparameter inspizieren

Die FIT-Konfiguration sammelt alle wichtigen Pipeline-Optionen. Die folgenden Zellen listen die wesentlichen Einstellungen für die Geschlechts- und Individual-ID-Modelle auf, damit Anpassungen bequem vorgenommen werden können.

In [ ]:
from pprint import pprint

sex_config = config.CONFIG['pipeline_sex']
id_config = config.CONFIG['pipeline_individual_id']

print('Sex pipeline defaults:')
pprint({
    'model_keys': sex_config['model_keys'],
    'search_spaces': {k: sex_config['search_spaces'][k] for k in ['outlier', 'scale', 'select__method', 'select__k', 'reduce_pre__method']},
    'metrics': sex_config['metrics'],
    'scoring': sex_config['scoring'],
})

print('Individual ID defaults:')
print(id_config)

Nach Wunsch können die Parameter direkt im Notebook überschrieben werden. Die folgenden Dictionaries bündeln die wichtigsten Optionen, sodass sie zentral angepasst werden können.

In [ ]:
# Ziel-Spezies definieren
TARGET_SPECIES = ['eurasian_otter']  # bei Bedarf erweitern

# Parameter für die Geschlechtsklassifikation
SEX_PIPELINE_PARAMS = {
    'model_keys': sex_config['model_keys'],
    'fs_method': 'random_forest',
    'fs_k': 12,
    'impute_method': None,
    'outlier_method': 'clip',
    'scaler_method': 'standard',
    'reduce_pre_method': None,
    'reduce_post_method': None,
    'n_jobs': -1,
    'debug': False,
}

# Optionen für die Individual-ID-Pipeline
ID_CONFIG_OVERRIDES = {
    'pairwise_defaults': {
        **id_config['pairwise_defaults'],
        'k_features': 24,
        'reducers': ['lda'],
        'selection_method': 'forward',
        'use_sexmodel_prediction': True,
    },
    'sequential_holdout_val_sizes': [2, 4, 6],
}
config.CONFIG['pipeline_individual_id'].update(ID_CONFIG_OVERRIDES)

SEX_PIPELINE_PARAMS

## 3. Geschlechtsmodelle trainieren

Der folgende Abschnitt bereitet zunächst die Datengrundlage vor (Import, Splitting, Zusammenfassungen) und startet anschließend das Training für alle konfigurierten Modelle.

In [ ]:
from FIT_python.pipeline_sex import PipelineWrapper

sex_wrapper = PipelineWrapper(**SEX_PIPELINE_PARAMS)
sex_wrapper.prepare()
sex_results = sex_wrapper.train()
sex_results.head()

## 4. Unsicherheitsanalyse für die Geschlechtsmodelle

Über `create_uncertainty_report` werden Kalibrierungs- und Vertrauensauswertungen erstellt. Zusätzlich zeigt das Notebook die automatisch erzeugten Reliability-Diagramme an.

In [ ]:
from IPython.display import display, Image
from FIT_python.pipeline_sex import create_uncertainty_report
from FIT_python.utils import get_species_paths

sex_uncertainty = {}
for species in TARGET_SPECIES:
    print(f"\n=== {species} ===")
    summary = create_uncertainty_report(
        species,
        bootstrap_iterations=200,
        bootstrap_confidence=0.95,
        random_state=config.GLOBAL_RANDOM_SEED,
        use_cv_train_predictions=True,
        create_plot=True,
    )
    sex_uncertainty[species] = summary
    display(summary['metrics'])
    display(summary['calibration'].head())
    paths = get_species_paths(species, section='sex_modelling')
    plot_path = paths['tables'] / 'uncertainty' / 'sex_uncertainty_reliability.png'
    if plot_path.exists():
        display(Image(filename=str(plot_path)))


## 5. Individual-ID-Pipelines trainieren

Die Individual-ID-Auswertung baut auf den vorbereiteten Splits sowie den Sex-Predictions auf. Das Training umfasst Pairwise-Modelle, UMAP-Embeddings und Siamese-Varianten.

In [ ]:
from FIT_python.pipeline_individual_id.train_individual_id_pipelines import main as train_id_pipelines

for species in TARGET_SPECIES:
    print(f"\nStarte Individual-ID-Training für {species!r}...")
    train_id_pipelines(species)


## 6. Unsicherheitsvisualisierung für die Individual-ID-Vorhersagen

Die Trainings-Skripte legen für jede Holdout-Runde Unsicherheitsberichte ab. Wir lesen die erzeugten CSV-Dateien ein, visualisieren die Kalibrierung und stellen die Vertrauensbänder als Balkendiagramm dar.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from FIT_python.utils import get_species_paths

for species in TARGET_SPECIES:
    print(f"\n=== {species} ===")
    paths = get_species_paths(species, section='individual_id')
    summary_root = paths['tables'] / 'individual_id_pipelines'
    for split_dir in sorted(summary_root.glob('split_*_umap_uncertainty')):
        metrics_fp = split_dir / 'pairwise_uncertainty_metrics.csv'
        calib_fp = split_dir / 'pairwise_uncertainty_calibration.csv'
        conf_fp = split_dir / 'pairwise_uncertainty_confidence_bands.csv'
        print(f"  -> {split_dir.name}")
        if metrics_fp.exists():
            display(pd.read_csv(metrics_fp))
        if calib_fp.exists():
            calib_df = pd.read_csv(calib_fp)
            if not calib_df.empty:
                fig, ax = plt.subplots(figsize=(6, 5))
                ax.scatter(calib_df['mean_predicted'], calib_df['fraction_of_positives'], s=60)
                ax.plot([0, 1], [0, 1], '--', color='black', linewidth=1)
                ax.set_title(f"Reliability – {species} ({split_dir.name})")
                ax.set_xlabel('Mean predicted probability')
                ax.set_ylabel('Fraction of positives')
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1)
                plt.show()
        if conf_fp.exists():
            conf_df = pd.read_csv(conf_fp)
            if not conf_df.empty:
                fig, ax = plt.subplots(figsize=(6, 4))
                sns.barplot(data=conf_df, x='confidence_band', y='fraction', ax=ax, color='#1f77b4')
                ax.set_title(f"Confidence bands – {species} ({split_dir.name})")
                ax.set_xlabel('Confidence band')
                ax.set_ylabel('Share of comparisons')
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()


## 7. Nächste Schritte

* Die erzeugten CSV-Dateien in `results/notebook_runs/<experiment>/results/...` können für weitergehende Analysen genutzt werden.
* Über die Parameter-Dictionaries lassen sich schnell alternative Modellkonfigurationen testen.
* Für automatisierte Reports können die hier gezeigten Funktionsaufrufe in Skripte übernommen werden.